In [1]:
import numpy as np

import sys
sys.path.insert(1,'C:\Zmyfiles\codes\research\pyDeepInsight_SRT\pyDeepInsight_SRT\examples\scripts')

from Models.Basic_kPCA import kernel_pca

In [2]:

if __name__ == "__main__":
    # Sample dataset (e.g., 4 samples, 2 features)
    X = np.array([[1, 2,3, 4], [5, 6,7, 8],[9,10,11,12]])

    # Perform kPCA with 2 components and gamma=0.5
    X_kpca = kernel_pca(X, n_components=3, gamma=0.5)
    print("Transformed data using Kernel PCA:\n", X_kpca)

Transformed data using Kernel PCA:
 [[-7.04936598e-01 -4.11984296e-01 -8.32667268e-17]
 [-4.32056755e-03  8.16485150e-01  1.66533454e-16]
 [ 7.09257165e-01 -4.04500853e-01 -5.55111512e-17]]


In [3]:
import pandas as pd
expr_file = r"..\data\Survival_TCGA_VS_Desmd\TCGA_BRCA_NvT.csv"
expr = pd.read_csv(expr_file, encoding='utf-8', engine='python')
expr.shape

(22490, 1212)

In [26]:
expr['TCGA-BH-A0B6-01A'][22488]

9.275500987

In [33]:
import re
new_values={}
for i in range(0,1212):
    s=expr.columns[i]
    match=re.search(r'-(\d{2})[AB]',s)
    if match:
        extracted_value = match.group(1)
        # if(extracted_value=='01'):
        #     expr[s][22489]="Tumor"
        # elif (extracted_value=='11'):
        #     expr[s][22489]=="Normal"
        # else:
        #     expr[s][22489]="Check"

        if(extracted_value=='01'):
           new_values[s]="Tumor"
        elif (extracted_value=='11'):
           new_values[s]="Normal"
        else:
            new_values[s]="Check"
expr.loc[expr.shape[0]] = new_values


In [4]:
import numpy as np

# Initialize variables
n_s, d_shared = 165, 50  # Source samples, shared dimensions
n_t, d_full = 112, 12000  # Target samples, full dimensions
lambda_reg = 0.1  # Regularization parameter
learning_rate = 0.01
iterations = 100

# Random initializations
Z_s_shared = np.random.rand(n_s, d_shared)  # Source shared projection
Z_t_shared = np.random.rand(n_t, d_shared)  # Target shared projection
Z_t = np.random.rand(n_t, d_full)  # Full target projection

# Compute MMD or discrepancy term
def compute_discrepancy(Z_s, Z_t):
    """Compute discrepancy (e.g., MMD, Frobenius norm) between Z_s and Z_t."""
    return np.linalg.norm(Z_s - Z_t, ord='fro')**2

# Iterative updates
for i in range(iterations):
    # Compute gradients
    # Gradient for the shared space term (simplified for Frobenius norm)
    grad_shared = 2 * (Z_t_shared - Z_s_shared) / n_t

    # Gradient for the full target space trace term
    grad_full = 2 * lambda_reg * Z_t

    # Update target projections
    Z_t_shared -= learning_rate * grad_shared  # Update shared space
    Z_t -= learning_rate * grad_full  # Update full target space

    # Loss (optional, for monitoring)
    discrepancy_loss = compute_discrepancy(Z_s_shared, Z_t_shared)
    trace_loss = lambda_reg * np.trace(Z_t.T @ Z_t)
    total_loss = discrepancy_loss + trace_loss
    print(f"Iteration {i + 1}, Total Loss: {total_loss:.4f}")

print("Final Shared Projection:", Z_t_shared)
print("Final Full Projection:", Z_t)


ValueError: operands could not be broadcast together with shapes (112,50) (165,50) 